# Advanced Problems: Python 3.6 Underscores in Numeric Literals

This notebook contains advanced practice problems with solutions for Python's underscore support in numeric literals and numeric formatting.

Topic focus:

- Valid and invalid underscore placement in numeric literals
- Decimal, binary, octal, hexadecimal, float, complex, and exponent notation
- String conversion using `int`, `float`, and `complex`
- Formatting numbers with `_` as a separator
- Writing robust validation and normalization utilities
- Avoiding common traps involving identifiers, prefixes, suffixes, and syntax errors

## Quick Reference

Python 3.6 introduced underscores in numeric literals.

Examples:

```python
1_000_000
0b_1010_1111
0o_755
0x_FF_FF
3.141_592
1.2e3_4
10_000 + 5_000j
```

General rule:

- Underscores may separate digits.
- They cannot appear at the start or end of a numeric literal.
- They cannot appear twice in a row.
- They cannot appear immediately before or after a decimal point.
- They cannot appear immediately before or after an exponent marker such as `e` or `E`.
- They may appear after a base prefix such as `0x`, `0b`, or `0o`.

String conversions such as `int('1_000')` and `float('1_000.5')` also understand underscores, but invalid placement still raises an error.

---

## Problem 1: Classify Numeric Literal Validity

For each expression below, predict whether it is valid Python syntax. If valid, evaluate it. If invalid, explain why.

```python
1_000_000
_1000
1000_
1__000
0x_FF_FF
0x__FF
0b_1010_0001
0o_755
3.141_592
3_.141
3._141
1.2e3_4
1.2_e34
1.2e_34
10_000j
10_000_j
```

Write code that safely tests these expressions without stopping the notebook when syntax errors occur.

In [1]:
expressions = [
    '1_000_000',
    '_1000',
    '1000_',
    '1__000',
    '0x_FF_FF',
    '0x__FF',
    '0b_1010_0001',
    '0o_755',
    '3.141_592',
    '3_.141',
    '3._141',
    '1.2e3_4',
    '1.2_e34',
    '1.2e_34',
    '10_000j',
    '10_000_j'
]

for expr in expressions:
    try:
        value = eval(expr)
        print(f'{expr:15} -> VALID   -> {value!r}')
    except SyntaxError as ex:
        print(f'{expr:15} -> INVALID -> SyntaxError: {ex.msg}')
    except Exception as ex:
        print(f'{expr:15} -> INVALID -> {type(ex).__name__}: {ex}')

1_000_000       -> VALID   -> 1000000
_1000           -> INVALID -> NameError: name '_1000' is not defined
1000_           -> INVALID -> SyntaxError: invalid decimal literal
1__000          -> INVALID -> SyntaxError: invalid decimal literal
0x_FF_FF        -> VALID   -> 65535
0x__FF          -> INVALID -> SyntaxError: invalid hexadecimal literal
0b_1010_0001    -> VALID   -> 161
0o_755          -> VALID   -> 493
3.141_592       -> VALID   -> 3.141592
3_.141          -> INVALID -> SyntaxError: invalid decimal literal
3._141          -> INVALID -> SyntaxError: invalid decimal literal
1.2e3_4         -> VALID   -> 1.2e+34
1.2_e34         -> INVALID -> SyntaxError: invalid decimal literal
1.2e_34         -> INVALID -> SyntaxError: invalid decimal literal
10_000j         -> VALID   -> 10000j
10_000_j        -> INVALID -> SyntaxError: invalid decimal literal


### Solution Explanation

Expected results:

| Expression | Result | Explanation |
|---|---:|---|
| `1_000_000` | Valid | Underscores separate decimal digits. |
| `_1000` | Valid expression, but not a numeric literal | This is an identifier name, not a number. It raises `NameError` unless defined. |
| `1000_` | Invalid | Literal cannot end with `_`. |
| `1__000` | Invalid | Consecutive underscores are not allowed. |
| `0x_FF_FF` | Valid | Underscore may appear after base prefix and between digits. |
| `0x__FF` | Invalid | Consecutive underscores after prefix are not allowed. |
| `0b_1010_0001` | Valid | Binary digits may be grouped. |
| `0o_755` | Valid | Octal digits may be grouped. |
| `3.141_592` | Valid | Fractional digits may be grouped. |
| `3_.141` | Invalid | Underscore cannot appear immediately before decimal point. |
| `3._141` | Invalid | Underscore cannot appear immediately after decimal point. |
| `1.2e3_4` | Valid | Exponent digits may be grouped. Equivalent to `1.2e34`. |
| `1.2_e34` | Invalid | Underscore cannot appear before exponent marker. |
| `1.2e_34` | Invalid | Underscore cannot appear immediately after exponent marker. |
| `10_000j` | Valid | Imaginary numeric literal. |
| `10_000_j` | Invalid | Underscore cannot appear before the imaginary suffix `j`. |

---

## Problem 2: Detect the Difference Between Identifiers and Numeric Literals

The expression `_1000` looks like a malformed numeric literal, but Python treats it as a variable name.

Create a function `classify_expression(expr)` that returns one of the following strings:

- `'numeric literal'`
- `'identifier'`
- `'valid expression but not a simple literal'`
- `'syntax error'`

Test it on:

```python
1_000
_1000
x + 1_000
1000_
0x_FF
True
None
3.14_15
```

Hint: use the `ast` module.

In [2]:
import ast

def classify_expression(expr):
    try:
        tree = ast.parse(expr, mode='eval')
    except SyntaxError:
        return 'syntax error'

    node = tree.body

    if isinstance(node, ast.Name):
        return 'identifier'

    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float, complex)):
        return 'numeric literal'

    return 'valid expression but not a simple literal'


tests = [
    '1_000',
    '_1000',
    'x + 1_000',
    '1000_',
    '0x_FF',
    'True',
    'None',
    '3.14_15'
]

for test in tests:
    print(f'{test:12} -> {classify_expression(test)}')

1_000        -> numeric literal
_1000        -> identifier
x + 1_000    -> valid expression but not a simple literal
1000_        -> syntax error
0x_FF        -> numeric literal
True         -> numeric literal
None         -> valid expression but not a simple literal
3.14_15      -> numeric literal


### Solution Explanation

`ast.parse(expr, mode='eval')` lets us parse an expression safely without evaluating it.

Important distinction:

```python
_1000
```

is not a malformed number. It is a valid identifier. It only fails at runtime if no variable named `_1000` exists.

By contrast:

```python
1000_
```

is a true syntax error because Python tries to parse it as a numeric literal ending with an underscore.

---

## Problem 3: Build a Validator for Decimal Integer Literals

Write a function `is_valid_decimal_integer_literal(s)` that returns `True` if `s` is a valid decimal integer literal using underscores and `False` otherwise.

This validator should accept:

```python
0
10
1_000
123_456_789
```

It should reject:

```python
_1000
1000_
1__000
12_34_'
1.000
0x_FF
```

Do not use `eval`.

In [3]:
import re

decimal_integer_pattern = re.compile(r'^[0-9](?:_?[0-9])*$')

def is_valid_decimal_integer_literal(s):
    return bool(decimal_integer_pattern.fullmatch(s))


valid_cases = ['0', '10', '1_000', '123_456_789']
invalid_cases = ['_1000', '1000_', '1__000', "12_34_'", '1.000', '0x_FF']

for case in valid_cases + invalid_cases:
    print(f'{case!r:15} -> {is_valid_decimal_integer_literal(case)}')

'0'             -> True
'10'            -> True
'1_000'         -> True
'123_456_789'   -> True
'_1000'         -> False
'1000_'         -> False
'1__000'        -> False
"12_34_'"       -> False
'1.000'         -> False
'0x_FF'         -> False


### Solution Explanation

The regular expression is:

```python
^[0-9](?:_?[0-9])*$
```

It means:

- Start with a digit.
- Then repeat zero or more groups consisting of an optional underscore followed by a digit.
- This prevents leading underscores, trailing underscores, and repeated underscores.

The function is intentionally limited to decimal integer literals. It rejects floats and prefixed bases such as hexadecimal, binary, and octal.

---

## Problem 4: Validate Binary, Octal, and Hexadecimal Integer Literals

Create a function `is_valid_prefixed_integer_literal(s)` that accepts valid binary, octal, and hexadecimal literals with underscores.

It should accept:

```python
0b1010
0b_1010_1111
0B_1010
0o755
0o_755
0xFF
0x_FF_FF
0X_DEAD_BEEF
```

It should reject:

```python
0b_102
0b__1010
0o_789
0x_FF_
0x__FF
0x_
x_FF
```

Do not use `eval`.

In [4]:
import re

prefixed_patterns = {
    'b': re.compile(r'^0[bB]_?[01](?:_?[01])*$'),
    'o': re.compile(r'^0[oO]_?[0-7](?:_?[0-7])*$'),
    'x': re.compile(r'^0[xX]_?[0-9a-fA-F](?:_?[0-9a-fA-F])*$')
}

def is_valid_prefixed_integer_literal(s):
    return any(pattern.fullmatch(s) for pattern in prefixed_patterns.values())


cases = [
    '0b1010', '0b_1010_1111', '0B_1010',
    '0o755', '0o_755',
    '0xFF', '0x_FF_FF', '0X_DEAD_BEEF',
    '0b_102', '0b__1010', '0o_789',
    '0x_FF_', '0x__FF', '0x_', 'x_FF'
]

for case in cases:
    print(f'{case:15} -> {is_valid_prefixed_integer_literal(case)}')

0b1010          -> True
0b_1010_1111    -> True
0B_1010         -> True
0o755           -> True
0o_755          -> True
0xFF            -> True
0x_FF_FF        -> True
0X_DEAD_BEEF    -> True
0b_102          -> False
0b__1010        -> False
0o_789          -> False
0x_FF_          -> False
0x__FF          -> False
0x_             -> False
x_FF            -> False


### Solution Explanation

Each base has its own valid digit set:

- Binary: `0`, `1`
- Octal: `0` through `7`
- Hexadecimal: `0` through `9`, `a` through `f`, `A` through `F`

The pattern allows one optional underscore immediately after the base prefix, as in:

```python
0x_FF
0b_1010
0o_755
```

But it does not allow:

```python
0x__FF
0x_
0x_FF_
```

---

## Problem 5: Normalize Numeric Strings Before Storing Them

Suppose a configuration file may contain integer values written with underscores:

```python
10_000
0x_FF_FF
0b_1010_0001
```

Write a function `parse_integer_config_value(s)` that:

1. Accepts decimal, binary, octal, and hexadecimal integer strings.
2. Allows underscores only where Python allows them.
3. Returns the integer value.
4. Raises `ValueError` with a helpful message for invalid input.

Do not use `eval`.

In [5]:
def parse_integer_config_value(s):
    if not isinstance(s, str):
        raise TypeError('configuration value must be a string')

    text = s.strip()

    if is_valid_decimal_integer_literal(text):
        return int(text, 10)

    if is_valid_prefixed_integer_literal(text):
        return int(text, 0)

    raise ValueError(f'invalid integer literal: {s!r}')


examples = [
    '10_000',
    '0x_FF_FF',
    '0b_1010_0001',
    '0o_755',
    '1__000',
    '0x__FF',
    '3.14'
]

for example in examples:
    try:
        print(f'{example:15} -> {parse_integer_config_value(example)}')
    except Exception as ex:
        print(f'{example:15} -> ERROR: {ex}')

10_000          -> 10000
0x_FF_FF        -> 65535
0b_1010_0001    -> 161
0o_755          -> 493
1__000          -> ERROR: invalid integer literal: '1__000'
0x__FF          -> ERROR: invalid integer literal: '0x__FF'
3.14            -> ERROR: invalid integer literal: '3.14'


### Solution Explanation

`int(text, 0)` is useful for prefixed integers because Python infers the base:

```python
int('0x_FF', 0)     # hexadecimal
int('0b_1010', 0)   # binary
int('0o_755', 0)    # octal
```

However, before converting, we validate the string to control error messages and make the accepted grammar explicit.

---

## Problem 6: Format Integers Using Underscore Separators

For the integer below:

```python
n = 3735928559
```

Produce all of the following strings:

```python
'3_735_928_559'
'1101_1110_1010_1101_1011_1110_1110_1111'
'3365_5335_7357'
'DEAD_BEEF'
'dead_beef'
```

Use format specifications, not manual string slicing.

In [6]:
n = 3735928559

decimal_form = f'{n:_}'
binary_form = f'{n:_b}'
octal_form = f'{n:_o}'
hex_upper_form = f'{n:_X}'
hex_lower_form = f'{n:_x}'

print(decimal_form)
print(binary_form)
print(octal_form)
print(hex_upper_form)
print(hex_lower_form)

3_735_928_559
1101_1110_1010_1101_1011_1110_1110_1111
336_5333_7357
DEAD_BEEF
dead_beef


### Solution Explanation

The `_` format option inserts separators:

- Decimal integers are grouped by thousands.
- Binary, octal, and hexadecimal are grouped every four digits.

Examples:

```python
f'{n:_}'
f'{n:_b}'
f'{n:_o}'
f'{n:_X}'
f'{n:_x}'
```

---

## Problem 7: Compare Comma and Underscore Formatting

Given:

```python
values = [10, 1000, 1000000, 1234567890, 12345.6789]
```

Print a table with three columns:

- Raw value
- Comma-formatted value
- Underscore-formatted value

For floats, show exactly two digits after the decimal point.

In [7]:
values = [10, 1000, 1000000, 1234567890, 12345.6789]

print(f'{'Raw':>15} | {'Comma':>15} | {'Underscore':>15}')
print('-' * 53)

for value in values:
    if isinstance(value, float):
        comma = f'{value:,.2f}'
        underscore = f'{value:_.2f}'
    else:
        comma = f'{value:,}'
        underscore = f'{value:_}'

    print(f'{str(value):>15} | {comma:>15} | {underscore:>15}')

            Raw |           Comma |      Underscore
-----------------------------------------------------
             10 |              10 |              10
           1000 |           1,000 |           1_000
        1000000 |       1,000,000 |       1_000_000
     1234567890 |   1,234,567,890 |   1_234_567_890
     12345.6789 |       12,345.68 |       12_345.68


### Solution Explanation

Python's format mini-language supports both comma and underscore separators:

```python
f'{1000000:,}'   # '1,000,000'
f'{1000000:_}'   # '1_000_000'
```

For floats:

```python
f'{12345.6789:_.2f}'
```

means:

- Use `_` as the thousands separator.
- Show exactly two digits after the decimal point.

---

## Problem 8: Parse Financial Amounts with Underscores

A system receives financial amounts as strings. It permits underscores for readability:

```python
1_000.00
12_345_678.90
0.99
```

Write `parse_money(s)` that:

1. Accepts only non-negative decimal amounts.
2. Requires exactly two digits after the decimal point.
3. Allows underscores in the integer part only.
4. Rejects underscores in the fractional part.
5. Returns the amount in cents as an integer.

Examples:

```python
parse_money('1_000.00') == 100000
parse_money('12_345_678.90') == 1234567890
parse_money('0.99') == 99
```

In [8]:
money_pattern = re.compile(r'^(?P<dollars>[0-9](?:_?[0-9])*)\.(?P<cents>[0-9]{2})$')

def parse_money(s):
    match = money_pattern.fullmatch(s)
    if not match:
        raise ValueError(f'invalid money amount: {s!r}')

    dollars = int(match.group('dollars').replace('_', ''))
    cents = int(match.group('cents'))
    return dollars * 100 + cents


tests = [
    '1_000.00',
    '12_345_678.90',
    '0.99',
    '1_000.0_0',
    '_100.00',
    '100_.00',
    '100.000',
    '-100.00'
]

for test in tests:
    try:
        print(f'{test:15} -> {parse_money(test)} cents')
    except ValueError as ex:
        print(f'{test:15} -> ERROR: {ex}')

1_000.00        -> 100000 cents
12_345_678.90   -> 1234567890 cents
0.99            -> 99 cents
1_000.0_0       -> ERROR: invalid money amount: '1_000.0_0'
_100.00         -> ERROR: invalid money amount: '_100.00'
100_.00         -> ERROR: invalid money amount: '100_.00'
100.000         -> ERROR: invalid money amount: '100.000'
-100.00         -> ERROR: invalid money amount: '-100.00'


### Solution Explanation

Even though Python allows underscores in floating-point strings such as:

```python
float('1_000.00')
```

financial parsing should usually avoid binary floating-point. Returning integer cents avoids precision problems.

This solution deliberately rejects values like:

```python
1_000.0_0
```

because the problem statement only permits underscores in the integer part.

---

## Problem 9: Understand Float and Exponent Underscore Rules

Predict which strings can be converted using `float(s)`.

```python
1_000.25
1_000._25
1_000.25_0
1_000.25_
1_000e2
1_000e2_5
1_000e_25
1_000_e25
```

Then write code to verify your predictions.

In [9]:
float_tests = [
    '1_000.25',
    '1_000._25',
    '1_000.25_0',
    '1_000.25_',
    '1_000e2',
    '1_000e2_5',
    '1_000e_25',
    '1_000_e25'
]

for s in float_tests:
    try:
        print(f'{s:12} -> {float(s)}')
    except ValueError as ex:
        print(f'{s:12} -> ERROR: {ex}')

1_000.25     -> 1000.25
1_000._25    -> ERROR: could not convert string to float: '1_000._25'
1_000.25_0   -> 1000.25
1_000.25_    -> ERROR: could not convert string to float: '1_000.25_'
1_000e2      -> 100000.0
1_000e2_5    -> 1e+28
1_000e_25    -> ERROR: could not convert string to float: '1_000e_25'
1_000_e25    -> ERROR: could not convert string to float: '1_000_e25'


### Solution Explanation

Valid:

```python
float('1_000.25')
float('1_000.25_0')
float('1_000e2')
float('1_000e2_5')
```

Invalid:

```python
float('1_000._25')
float('1_000.25_')
float('1_000e_25')
float('1_000_e25')
```

Underscores may separate digits, including fractional digits and exponent digits. They may not touch the decimal point or exponent marker.

---

## Problem 10: Write a Safe Calculator for Numeric Literals Only

Create a function `safe_add_literals(a, b)` that accepts two strings representing numeric literals and returns their sum.

Requirements:

- Accept valid Python integer and float literals with underscores.
- Accept binary, octal, and hexadecimal integers.
- Reject arbitrary expressions like `'1 + 2'`.
- Reject identifiers like `'_1000'`.
- Do not use `eval`.

Examples:

```python
safe_add_literals('1_000', '2_000') == 3000
safe_add_literals('0x_FF', '1') == 256
safe_add_literals('1_000.5', '0.25') == 1000.75
```

In [10]:
def parse_numeric_literal_only(s):
    if not isinstance(s, str):
        raise TypeError('literal must be a string')

    text = s.strip()

    try:
        tree = ast.parse(text, mode='eval')
    except SyntaxError:
        raise ValueError(f'invalid numeric literal: {s!r}') from None

    node = tree.body

    if not (isinstance(node, ast.Constant) and isinstance(node.value, (int, float))):
        raise ValueError(f'not a simple numeric literal: {s!r}')

    return node.value


def safe_add_literals(a, b):
    return parse_numeric_literal_only(a) + parse_numeric_literal_only(b)


tests = [
    ('1_000', '2_000'),
    ('0x_FF', '1'),
    ('1_000.5', '0.25'),
    ('1 + 2', '3'),
    ('_1000', '1'),
    ('1__000', '1')
]

for a, b in tests:
    try:
        print(f'{a!r} + {b!r} -> {safe_add_literals(a, b)}')
    except Exception as ex:
        print(f'{a!r} + {b!r} -> ERROR: {ex}')

'1_000' + '2_000' -> 3000
'0x_FF' + '1' -> 256
'1_000.5' + '0.25' -> 1000.75
'1 + 2' + '3' -> ERROR: not a simple numeric literal: '1 + 2'
'_1000' + '1' -> ERROR: not a simple numeric literal: '_1000'
'1__000' + '1' -> ERROR: invalid numeric literal: '1__000'


### Solution Explanation

Using `ast.parse` allows Python itself to parse the literal grammar, including underscore rules, without executing arbitrary code.

The function rejects expressions like:

```python
1 + 2
```

because the AST node is a binary operation, not a simple numeric constant.

It also rejects:

```python
_1000
```

because that is an identifier, not a numeric literal.

---

## Problem 11: Explain Why `int()` and `eval()` Behave Differently

Predict the output of each expression:

```python
int('1_000')
eval('1_000')
int('_1000')
eval('_1000')
int('0x_FF', 0)
eval('0x_FF')
int('0x__FF', 0)
eval('0x__FF')
```

Then verify your predictions.

In [11]:
comparisons = [
    "int('1_000')",
    "eval('1_000')",
    "int('_1000')",
    "eval('_1000')",
    "int('0x_FF', 0)",
    "eval('0x_FF')",
    "int('0x__FF', 0)",
    "eval('0x__FF')"
]

for expr in comparisons:
    try:
        print(f'{expr:22} -> {eval(expr)!r}')
    except Exception as ex:
        print(f'{expr:22} -> ERROR: {type(ex).__name__}: {ex}')

int('1_000')           -> 1000
eval('1_000')          -> 1000
int('_1000')           -> ERROR: ValueError: invalid literal for int() with base 10: '_1000'
eval('_1000')          -> ERROR: NameError: name '_1000' is not defined
int('0x_FF', 0)        -> 255
eval('0x_FF')          -> 255
int('0x__FF', 0)       -> ERROR: ValueError: invalid literal for int() with base 0: '0x__FF'
eval('0x__FF')         -> ERROR: SyntaxError: invalid hexadecimal literal (<string>, line 1)


### Solution Explanation

`int('1_000')` parses a string as an integer and allows underscores in valid positions.

`eval('1_000')` asks Python to parse and evaluate a Python expression. Since `1_000` is a valid numeric literal, it succeeds.

`int('_1000')` fails because `_1000` is not a valid integer string.

`eval('_1000')` parses successfully because `_1000` is an identifier, but then it usually raises `NameError` because no such variable exists.

`int('0x_FF', 0)` and `eval('0x_FF')` both succeed.

`0x__FF` is invalid because it has consecutive underscores after the base prefix.

---

## Problem 12: Implement a Readable Memory Size Parser

Write `parse_memory_size(s)` that parses strings such as:

```python
1_024B
64_KB
1_5MB
2GB
```

Rules:

- The numeric part must be a valid decimal integer literal with underscores.
- Units are `B`, `KB`, `MB`, and `GB`.
- Unit multipliers use powers of 1024.
- Return the size in bytes as an integer.
- Reject invalid underscore placement.

Examples:

```python
parse_memory_size('1_024B') == 1024
parse_memory_size('64_KB') == 65536
parse_memory_size('1_5MB') == 15728640
parse_memory_size('2GB') == 2147483648
```

In [12]:
memory_pattern = re.compile(r'^(?P<number>[0-9](?:_?[0-9])*)(?P<unit>B|KB|MB|GB)$')

def parse_memory_size(s):
    match = memory_pattern.fullmatch(s)
    if not match:
        raise ValueError(f'invalid memory size: {s!r}')

    number = int(match.group('number').replace('_', ''))
    unit = match.group('unit')

    multipliers = {
        'B': 1,
        'KB': 1024,
        'MB': 1024 ** 2,
        'GB': 1024 ** 3
    }

    return number * multipliers[unit]


tests = ['1_024B', '64_KB', '1_5MB', '2GB', '_64KB', '64_K_B', '64__KB', '64KB_']

for test in tests:
    try:
        print(f'{test:10} -> {parse_memory_size(test)}')
    except ValueError as ex:
        print(f'{test:10} -> ERROR: {ex}')

1_024B     -> 1024
64_KB      -> ERROR: invalid memory size: '64_KB'
1_5MB      -> 15728640
2GB        -> 2147483648
_64KB      -> ERROR: invalid memory size: '_64KB'
64_K_B     -> ERROR: invalid memory size: '64_K_B'
64__KB     -> ERROR: invalid memory size: '64__KB'
64KB_      -> ERROR: invalid memory size: '64KB_'


### Solution Explanation

The numeric part is validated with the same decimal integer rule:

```python
[0-9](?:_?[0-9])*
```

This accepts unusual but legal groupings like:

```python
1_5MB
```

because Python does not require groups of exactly three digits. Underscores are readability separators, not thousands-group validators.

---

## Problem 13: Convert Between Grouping Styles

Write a function `regroup_integer_string(s, base=10, group_size=None)` that:

1. Accepts a valid integer string with underscores.
2. Converts it to an integer.
3. Re-formats it using Python's underscore grouping.

Rules:

- For base `10`, use decimal underscore formatting.
- For base `2`, use binary underscore formatting.
- For base `8`, use octal underscore formatting.
- For base `16`, use lowercase hexadecimal underscore formatting.
- Reject unsupported bases.

Examples:

```python
regroup_integer_string('12_34_56', 10) == '123_456'
regroup_integer_string('0x_F_F_F_F', 16) == 'ffff'
regroup_integer_string('0b_11111111', 2) == '1111_1111'
```

In [13]:
def regroup_integer_string(s, base=10):
    if base not in {2, 8, 10, 16}:
        raise ValueError('base must be one of 2, 8, 10, or 16')

    text = s.strip()

    if base == 10:
        if not is_valid_decimal_integer_literal(text):
            raise ValueError(f'invalid decimal integer literal: {s!r}')
        value = int(text, 10)
        return f'{value:_}'

    if base == 2:
        if not re.fullmatch(r'^(?:0[bB]_?)?[01](?:_?[01])*$', text):
            raise ValueError(f'invalid binary integer literal: {s!r}')
        value = int(text, 0 if text.lower().startswith('0b') else 2)
        return f'{value:_b}'

    if base == 8:
        if not re.fullmatch(r'^(?:0[oO]_?)?[0-7](?:_?[0-7])*$', text):
            raise ValueError(f'invalid octal integer literal: {s!r}')
        value = int(text, 0 if text.lower().startswith('0o') else 8)
        return f'{value:_o}'

    if base == 16:
        if not re.fullmatch(r'^(?:0[xX]_?)?[0-9a-fA-F](?:_?[0-9a-fA-F])*$', text):
            raise ValueError(f'invalid hexadecimal integer literal: {s!r}')
        value = int(text, 0 if text.lower().startswith('0x') else 16)
        return f'{value:_x}'


tests = [
    ('12_34_56', 10),
    ('0x_F_F_F_F', 16),
    ('0b_11111111', 2),
    ('0o_777777', 8),
    ('1__234', 10)
]

for text, base in tests:
    try:
        print(f'{text:14} base {base:2} -> {regroup_integer_string(text, base)}')
    except ValueError as ex:
        print(f'{text:14} base {base:2} -> ERROR: {ex}')

12_34_56       base 10 -> 123_456
0x_F_F_F_F     base 16 -> ffff
0b_11111111    base  2 -> 1111_1111
0o_777777      base  8 -> 77_7777
1__234         base 10 -> ERROR: invalid decimal integer literal: '1__234'


### Solution Explanation

Python permits many grouping styles in input:

```python
12_34_56
0x_F_F_F_F
```

But formatting with `_` produces Python's standard grouping:

- Decimal: groups of three digits.
- Binary, octal, hexadecimal: groups of four digits.

This makes the function useful for normalization.

---

## Problem 14: Create a Linter for Suspicious Numeric Grouping

Python allows this:

```python
1_23_456
```

But many teams prefer decimal numbers to use thousands-style grouping:

```python
123_456
1_234_567
```

Write `has_standard_decimal_grouping(s)` that returns `True` only when:

1. `s` is a valid decimal integer literal.
2. If underscores are present, all groups after the first contain exactly three digits.
3. The first group contains one to three digits.

Examples:

```python
has_standard_decimal_grouping('1000') == True
has_standard_decimal_grouping('1_000') == True
has_standard_decimal_grouping('12_345_678') == True
has_standard_decimal_grouping('1234_567') == False
has_standard_decimal_grouping('1_23_456') == False
```

In [14]:
standard_decimal_grouping_pattern = re.compile(r'^(?:[0-9]+|[0-9]{1,3}(?:_[0-9]{3})+)$')

def has_standard_decimal_grouping(s):
    if not is_valid_decimal_integer_literal(s):
        return False
    return bool(standard_decimal_grouping_pattern.fullmatch(s))


tests = [
    '1000',
    '1_000',
    '12_345_678',
    '1234_567',
    '1_23_456',
    '123_456',
    '_123',
    '123_'
]

for test in tests:
    print(f'{test:12} -> {has_standard_decimal_grouping(test)}')

1000         -> True
1_000        -> True
12_345_678   -> True
1234_567     -> False
1_23_456     -> False
123_456      -> True
_123         -> False
123_         -> False


### Solution Explanation

This problem separates Python validity from style-guide validity.

Python accepts:

```python
1_23_456
```

because underscores only need to separate digits.

A team style guide may still reject it because it is not standard thousands grouping.

---

## Problem 15: Build a Numeric Literal Report from Source Code

Given a Python source-code string, write a function `numeric_literal_report(source)` that reports each numeric literal's original text and value.

Example input:

```python
source = '''
a = 1_000
b = 0x_FF
c = 3.14_15
d = 10_000j
e = a + b
'''
```

Expected output should include something like:

```python
1_000      -> 1000
0x_FF      -> 255
3.14_15    -> 3.1415
10_000j    -> 10000j
```

Hint: use `tokenize`, not `ast`, if you want the original literal text.

In [15]:
import io
import tokenize

def numeric_literal_report(source):
    report = []
    reader = io.StringIO(source).readline

    for token in tokenize.generate_tokens(reader):
        if token.type == tokenize.NUMBER:
            literal_text = token.string
            try:
                value = ast.literal_eval(literal_text)
            except Exception as ex:
                value = f'ERROR: {ex}'
            report.append((literal_text, value))

    return report


source = '''
a = 1_000
b = 0x_FF
c = 3.14_15
d = 10_000j
e = a + b
'''

for literal, value in numeric_literal_report(source):
    print(f'{literal:10} -> {value!r}')

1_000      -> 1000
0x_FF      -> 255
3.14_15    -> 3.1415
10_000j    -> 10000j


### Solution Explanation

`ast` is excellent for understanding parsed program structure, but it does not preserve the original formatting of numeric literals.

For example, both of these become the same integer value:

```python
1000
1_000
```

The `tokenize` module lets us inspect the original token text before Python normalizes it into a value.

---

## Problem 16: Identify Suspicious Numeric Literals in Source Code

Write a function `find_nonstandard_decimal_grouping(source)` that scans Python source code and returns decimal integer literals that are valid Python but do not follow standard thousands grouping.

Example:

```python
source = '''
a = 1000
b = 1_000
c = 12_345_678
d = 1_23_456
e = 1234_567
f = 0x_FF_FF
'''
```

Expected suspicious literals:

```python
1_23_456
1234_567
```

Hexadecimal, binary, octal, floats, and complex numbers should be ignored.

In [16]:
def find_nonstandard_decimal_grouping(source):
    suspicious = []
    reader = io.StringIO(source).readline

    for token in tokenize.generate_tokens(reader):
        if token.type != tokenize.NUMBER:
            continue

        text = token.string

        # Ignore non-decimal integers and non-integers.
        if text.lower().startswith(('0x', '0b', '0o')):
            continue
        if any(ch in text.lower() for ch in '.ej'):
            continue

        if '_' in text and is_valid_decimal_integer_literal(text):
            if not has_standard_decimal_grouping(text):
                suspicious.append((text, token.start))

    return suspicious


source = '''
a = 1000
b = 1_000
c = 12_345_678
d = 1_23_456
e = 1234_567
f = 0x_FF_FF
g = 3.14_15
h = 10_000j
'''

for literal, position in find_nonstandard_decimal_grouping(source):
    print(f'{literal:12} at line {position[0]}, column {position[1]}')

1_23_456     at line 5, column 4
1234_567     at line 6, column 4


### Solution Explanation

This is a realistic linter-style task.

The implementation:

1. Uses `tokenize` to find numeric tokens.
2. Ignores prefixed bases, floats, exponents, and imaginary numbers.
3. Looks only at decimal integers containing underscores.
4. Flags valid Python literals that violate a stricter style rule.

This shows an important best practice: Python syntax validity and project style validity are different concerns.

---

## Problem 17: Round-Trip Test Numeric Literal Formatting

Write a function `round_trip_integer(n, format_spec)` that:

1. Formats integer `n` using `format(n, format_spec)`.
2. Parses the formatted result back into an integer.
3. Returns `True` if the parsed value equals the original value.

Test these format specs:

```python
'_'
'_b'
'_o'
'_x'
'_X'
```

Use `n = 3735928559`.

In [17]:
def round_trip_integer(n, format_spec):
    formatted = format(n, format_spec)

    if format_spec.endswith(('b', 'o', 'x', 'X')):
        base_map = {'b': 2, 'o': 8, 'x': 16, 'X': 16}
        base = base_map[format_spec[-1]]
        parsed = int(formatted, base)
    else:
        parsed = int(formatted, 10)

    return formatted, parsed == n


n = 3735928559
for spec in ['_', '_b', '_o', '_x', '_X']:
    formatted, ok = round_trip_integer(n, spec)
    print(f'{spec:3} -> {formatted:40} -> round trip: {ok}')

_   -> 3_735_928_559                            -> round trip: True
_b  -> 1101_1110_1010_1101_1011_1110_1110_1111  -> round trip: True
_o  -> 336_5333_7357                            -> round trip: True
_x  -> dead_beef                                -> round trip: True
_X  -> DEAD_BEEF                                -> round trip: True


### Solution Explanation

`int()` can parse strings containing valid underscore separators.

Therefore these round trips are valid:

```python
int(format(n, '_'), 10)
int(format(n, '_b'), 2)
int(format(n, '_o'), 8)
int(format(n, '_x'), 16)
```

This is useful when generating readable machine-consumable output.

---

## Problem 18: Convert Integers to Python-Style Literals

Write `to_python_literal(n, base)` that returns a Python-style integer literal string with underscore grouping.

Rules:

- `base=2` returns strings like `'0b_1111_0000'`.
- `base=8` returns strings like `'0o_7777'`.
- `base=10` returns strings like `'1_000_000'`.
- `base=16` returns strings like `'0x_DEAD_BEEF'`.
- Negative numbers should preserve the sign.

Examples:

```python
to_python_literal(255, 2) == '0b_1111_1111'
to_python_literal(65535, 16) == '0x_FFFF'
to_python_literal(-1000000, 10) == '-1_000_000'
```

In [18]:
def to_python_literal(n, base):
    if not isinstance(n, int):
        raise TypeError('n must be an integer')

    sign = '-' if n < 0 else ''
    magnitude = abs(n)

    if base == 2:
        return sign + '0b_' + format(magnitude, '_b')
    if base == 8:
        return sign + '0o_' + format(magnitude, '_o')
    if base == 10:
        return sign + format(magnitude, '_')
    if base == 16:
        return sign + '0x_' + format(magnitude, '_X')

    raise ValueError('base must be one of 2, 8, 10, or 16')


tests = [
    (255, 2),
    (65535, 16),
    (-1000000, 10),
    (3735928559, 16),
    (511, 8)
]

for n, base in tests:
    print(f'{n:12} base {base:2} -> {to_python_literal(n, base)}')

         255 base  2 -> 0b_1111_1111
       65535 base 16 -> 0x_FFFF
    -1000000 base 10 -> -1_000_000
  3735928559 base 16 -> 0x_DEAD_BEEF
         511 base  8 -> 0o_777


### Solution Explanation

Python accepts an underscore immediately after a base prefix:

```python
0b_1111_1111
0o_7777
0x_DEAD_BEEF
```

For decimal numbers, no prefix is used.

Negative numeric literals are slightly subtle: syntactically, `-1_000` is parsed as unary minus applied to the positive literal `1_000`. For practical formatting, returning `'-1_000'` is still the expected readable representation.

---

## Problem 19: Validate Complex Number Strings with Underscores

Write a function `parse_imaginary_literal(s)` that accepts simple imaginary literals such as:

```python
1_000j
3.14_15j
1.2e3_4j
```

and returns a complex number.

It should reject:

```python
1_000_j
_1000j
1__000j
1.2e_34j
j
```

Use parsing rather than hand-written complex-number evaluation.

In [19]:
def parse_imaginary_literal(s):
    text = s.strip()

    try:
        tree = ast.parse(text, mode='eval')
    except SyntaxError:
        raise ValueError(f'invalid imaginary literal: {s!r}') from None

    node = tree.body

    if isinstance(node, ast.Constant) and isinstance(node.value, complex):
        return node.value

    raise ValueError(f'not a simple imaginary literal: {s!r}')


tests = [
    '1_000j',
    '3.14_15j',
    '1.2e3_4j',
    '1_000_j',
    '_1000j',
    '1__000j',
    '1.2e_34j',
    'j'
]

for test in tests:
    try:
        print(f'{test:12} -> {parse_imaginary_literal(test)}')
    except ValueError as ex:
        print(f'{test:12} -> ERROR: {ex}')

1_000j       -> 1000j
3.14_15j     -> 3.1415j
1.2e3_4j     -> 1.2e+34j
1_000_j      -> ERROR: invalid imaginary literal: '1_000_j'
_1000j       -> ERROR: not a simple imaginary literal: '_1000j'
1__000j      -> ERROR: invalid imaginary literal: '1__000j'
1.2e_34j     -> ERROR: invalid imaginary literal: '1.2e_34j'
j            -> ERROR: not a simple imaginary literal: 'j'


### Solution Explanation

This solution lets Python's parser decide whether the imaginary literal is valid.

Examples of valid imaginary literals:

```python
1_000j
3.14_15j
1.2e3_4j
```

Invalid examples:

```python
1_000_j
1.2e_34j
```

The underscore must separate digits. It cannot appear before the `j` suffix or immediately after the exponent marker.

---

## Problem 20: Design a Robust Literal Sanitizer

Write a function `sanitize_numeric_literal(s)` that prepares a numeric literal string for storage in a normalized form.

Requirements:

- Accept decimal integers, binary integers, octal integers, hexadecimal integers, floats, and imaginary literals.
- Reject identifiers and arbitrary expressions.
- Return a dictionary with:
  - `'original'`
  - `'kind'`
  - `'value'`
  - `'normalized'`

Kinds should be:

- `'int'`
- `'float'`
- `'complex'`

For integers, normalize using decimal underscore grouping.
For floats and complex numbers, normalize by removing underscores from the original literal.

Examples:

```python
sanitize_numeric_literal('1_000')
sanitize_numeric_literal('0x_FF')
sanitize_numeric_literal('3.14_15')
sanitize_numeric_literal('1_000j')
```

In [20]:
def sanitize_numeric_literal(s):
    if not isinstance(s, str):
        raise TypeError('input must be a string')

    text = s.strip()

    try:
        tree = ast.parse(text, mode='eval')
    except SyntaxError:
        raise ValueError(f'invalid numeric literal: {s!r}') from None

    node = tree.body

    if not isinstance(node, ast.Constant) or not isinstance(node.value, (int, float, complex)):
        raise ValueError(f'not a simple numeric literal: {s!r}')

    value = node.value

    if isinstance(value, bool):
        raise ValueError(f'not a numeric literal for this sanitizer: {s!r}')

    if isinstance(value, int):
        kind = 'int'
        normalized = f'{value:_}'
    elif isinstance(value, float):
        kind = 'float'
        normalized = text.replace('_', '')
    else:
        kind = 'complex'
        normalized = text.replace('_', '')

    return {
        'original': s,
        'kind': kind,
        'value': value,
        'normalized': normalized
    }


tests = [
    '1_000',
    '0x_FF',
    '3.14_15',
    '1_000j',
    '_1000',
    '1 + 2',
    '1__000'
]

for test in tests:
    try:
        print(sanitize_numeric_literal(test))
    except Exception as ex:
        print(f'{test!r} -> ERROR: {ex}')

{'original': '1_000', 'kind': 'int', 'value': 1000, 'normalized': '1_000'}
{'original': '0x_FF', 'kind': 'int', 'value': 255, 'normalized': '255'}
{'original': '3.14_15', 'kind': 'float', 'value': 3.1415, 'normalized': '3.1415'}
{'original': '1_000j', 'kind': 'complex', 'value': 1000j, 'normalized': '1000j'}
'_1000' -> ERROR: not a simple numeric literal: '_1000'
'1 + 2' -> ERROR: not a simple numeric literal: '1 + 2'
'1__000' -> ERROR: invalid numeric literal: '1__000'


### Solution Explanation

This final problem combines several best practices:

- Use `ast.parse`, not `eval`, when validating Python syntax.
- Accept only simple constants, not arbitrary expressions.
- Treat identifiers like `_1000` as invalid for numeric input.
- Normalize integer output using Python's own underscore formatting.
- Normalize float and complex strings carefully, avoiding unnecessary binary floating-point reformatting.

The function is intentionally conservative and suitable for configuration parsing, linting, and educational tooling.

---

# Summary

Key lessons:

1. Underscores in numeric literals are readability separators.
2. Python does not require thousands-style grouping.
3. Underscores must separate digits.
4. They cannot appear at the start, end, or next to punctuation such as `.`, `e`, `E`, or `j`.
5. Base prefixes may be followed by an underscore, as in `0x_FF`.
6. Formatting with `_` is useful for producing readable integers.
7. `tokenize` preserves original literal text; `ast` gives parsed structure and values.
8. Avoid `eval` for user input. Use `ast`, `int`, `float`, `complex`, regular expressions, or purpose-built parsers instead.